# 03 Gold Tables (A): Order flow

**Owner:** Person A
**Builds:** `gold.dim_customers`, `gold.fact_orders`, `gold.fact_order_items`, `gold.mart_customer_rfm`
**Rule:** joins happen only in gold, never in silver. Aggregate one-to-many tables to one row per `order_id` BEFORE joining.

## Build order
1. `dim_customers` (no dependencies)
2. `fact_order_items`
3. `fact_orders` (Person B depends on it, so tell B when it is ready)
4. `mart_customer_rfm` (needs all three above)

---

## 1. `gold.dim_customers`
- **Grain:** one row per `customer_id`.
- **Base:** `silver.customers`
- **Joins:** none
- **Columns:** `customer_id`, `customer_unique_id`, `zip_code_prefix`, `city`, `state`.
- **Note:** a real person is `customer_unique_id` (one person can have several `customer_id` values). Use `DISTINCTCOUNT(customer_unique_id)` in Power BI for customer counts.
- **Checks:** `customer_id` unique; row count equals `silver.customers`.

## 2. `gold.fact_orders`
- **Grain:** one row per order.
- **Base:** `silver.orders`
- **Joins:**

| Join | Type | Aggregate first (one row per `order_id`) | Columns produced |
|---|---|---|---|
| `silver.order_payments` | `LEFT JOIN` on `order_id` | `SUM(payment_value)`, `MAX(payment_installments)`, main payment type (the type with the highest value) | `total_payment`, `payment_installments_max`, `main_payment_type` |

- **Columns:** `order_id`, `customer_id`, `order_status`, `order_date` (DATE), `approved_date`, `delivered_carrier_date`, `delivered_customer_date`, `estimated_delivery_date`, `delivery_days`, `delay_days`, `is_late` (BIT), `total_payment`, `payment_installments_max`, `main_payment_type`.
- **Logic:**
  - Aggregate `silver.order_payments` by `order_id` first (sum of payment value, main payment type).
  - `delivery_days` = delivered date minus purchase date; `delay_days` = delivered date minus estimated date; `is_late` = `delay_days > 0`.
  - Delivery columns stay `NULL` for orders that were not delivered (`is_late` is NULL, not 0).
- **Checks:** `order_id` unique; row count equals `silver.orders`; total payment matches the sum in `silver.order_payments` (explain small differences in the log).

## 3. `gold.fact_order_items`
- **Grain:** one row per order item (`order_id` + `order_item_id`).
- **Base:** `silver.order_items`
- **Joins:**

| Join | Type | Why | Columns produced |
|---|---|---|---|
| `silver.orders` | `LEFT JOIN` on `order_id` | Orders is one row per order, so no duplication | `customer_id`, `order_date`, `order_status` |

- **Do NOT join** products, sellers or payments. Keep `product_id` and `seller_id` as keys; Power BI relates them to the dimensions.
- **Columns:** `order_id`, `order_item_id`, `product_id`, `seller_id`, `customer_id`, `order_date`, `order_status`, `price`, `freight_value`, `item_total` (price + freight).
- **Note:** keep payments out of this table. Order-level amounts live in `fact_orders`, otherwise they repeat on every item and inflate totals.
- **Checks:** row count equals `silver.order_items`; `SUM(price)` equals the silver total.

## 4. `gold.mart_customer_rfm`
- **Grain:** one row per `customer_unique_id`.
- **Base:** `gold.fact_orders`
- **Joins:**

| Join | Type | Aggregate first | Purpose |
|---|---|---|---|
| `gold.dim_customers` on `customer_id` | `JOIN` | none | maps each order to `customer_unique_id` |
| `gold.fact_order_items` on `order_id` | `LEFT JOIN` | `SUM(price)` per `order_id` | order value for `monetary` |

  Aggregate items to one row per order BEFORE joining to `fact_orders`, then group by `customer_unique_id`.
- **Columns:** `customer_unique_id`, `recency_days`, `frequency` (distinct orders), `monetary` (sum of item prices), `r_score`, `f_score`, `m_score`, `segment`.
- **Logic:** reference date is the day after the last order in the data (not today). Scores 1-5 with `NTILE`/`qcut`; most customers order only once, so frequency scoring needs a simple rule (for example 1 order, 2 orders, 3+ orders). Document the segment rules.
- **Why a mart:** segmentation is awkward in DAX, so it is calculated once here.
- **Checks:** `customer_unique_id` unique; row count equals `COUNT(DISTINCT customer_unique_id)` in `dim_customers`; all scores are between 1 and 5; `SUM(monetary)` is close to `SUM(price)` in `fact_order_items` (the difference comes from excluded statuses, if any).

## Validation (run at the end)
- Row counts: gold vs silver for every table above.
- No duplicate keys.
- Orphan check: every `fact_orders.customer_id` and `fact_order_items.customer_id` exists in `dim_customers`.